##### Observer Pattern

Suppose we have a financial tracker system that includes a dashboard, data logger and mobile sms alert tracking multiple stock and crypto prices. 

How do we update those 3 different components when stock price changes?

> Option 1: Polling 
- the dashboard, logger and SMS systems can periodically ask the price ticker "did the price change"? 
- but this waste resource since each system needs to call the tracker individually.

> Option 2: Hardcoading the components into the price tracker
- this works but if we want to add an Email alert component or remove SMS notifier tomorrow, we'd have to modify the core price tracker. This violates Open/Closed principle and single responsibility principles. The price tracker would have to know how to draw a line chart as well as how to send SMS and do the logging.

Both options above are bad. In this scenario, we should use the 
##### Publisher - Subscriber Model

The Observer pattern flips the dependency. The core price tracker becomes a Publisher (often called the Subject). It maintains a list of Subscribers (often called Observers).

When the price changes, the Publisher simply loops through its list and calls a standardized update() method on every Subscriber. The Publisher doesn't know what the Subscribers are doing with the data; it only knows they promised to implement the update() interface.

- subscription mechanism
    - array field for storing list of reference to subscriber objects
    - public methods for adding / removing subscribers
* A publisher might not know what type different subscribers will come in the future. So it is important that all subscribers have the same interface that publisher can use.

**The Components**

- *Observer (Interface)*: Defines the update() method. All subscribers must implement this.
- *Subject (Interface)*: Defines methods to attach(), detach(), and notify() observers.
- *Concrete Subject*: The actual publisher holding the state (e.g., the Stock Ticker).
- *Concrete Observers*: The actual subscribers doing the work (e.g., the Logger, the Chart).

In [8]:
from abc import ABC, abstractmethod
from typing import List

class Observer(ABC):
    """
    The Subscriber interface. 
    Every subscriber MUST implement this so the Publisher knows how to talk to it.
    """
    @abstractmethod
    def update(self, price : float) -> None:
        pass

class Subject(ABC):
    """
    The Publisher interface.
    Manages the subscription list and broadcasts events.
    """
    @abstractmethod
    def attach(self, observer: Observer) -> None:
        pass

    @abstractmethod
    def detach(self, observer: Observer) -> None:
        pass

    @abstractmethod
    def notify(self) -> None:
        pass

In [9]:
# --- the concrete publisher ----
class CryptoTicker(Subject):
    """
    The Publisher. It holds the core state (price) and the list of subscribers.
    """
    def __init__(self, symbol: str):
        self.symbol = symbol
        self._price = 0.0
        self._observer: List[Observer] = [] # subscription list

    def attach(self, observer: Observer) -> None:
        if observer not in self._observer:
            self._observer.append(observer)
    
    def detach(self, observer: Observer) -> None:
        if observer in self._observer:
            self._observer.remove(observer)
    
    def notify(self) -> None:
        """ Broadcast the state change to all subscribers. """
        for observer in self._observer:
            observer.update(self._price)
    
    def set_price(self, new_price: float) -> None:
        """Core business logic that triggers the notification."""
        print(f"\n[Ticker] {self.symbol} price updated to ${new_price}")
        self._price = new_price
        self.notify()  # Push the update to everyone listening

In [10]:
# --- The Concrete Subscribers ---
class TerminalChart(Observer):
    def update(self, price: float) -> None:
        print(f"-> [Chart] Rendering new candle at ${price}")

class DatabaseLogger(Observer):
    def update(self, price: float) -> None:
        print(f"   -> [Logger] INSERT INTO prices VALUES ({price})")

class WhaleAlertSystem(Observer):
    def update(self, price: float) -> None:
        if price > 50000:
            print(f"   -> [SMS Alert] WHALE ACTIVITY! Price breached $50k (${price})")

In [11]:
# --- Usage ---

# 1. Initialize the Publisher
bitcoin_ticker = CryptoTicker("BTC")

# 2. Initialize the Subscribers
chart = TerminalChart()
logger = DatabaseLogger()
alert = WhaleAlertSystem()

# 3. Wire them together (Subscription)
bitcoin_ticker.attach(chart)
bitcoin_ticker.attach(logger)
bitcoin_ticker.attach(alert)

# 4. Simulate State Changes
bitcoin_ticker.set_price(48000.00)
bitcoin_ticker.set_price(51200.50) # The alert system will trigger here!

# 5. Dynamic Unsubscription
print("\n[System] Disconnecting Logger for maintenance...")
bitcoin_ticker.detach(logger)

bitcoin_ticker.set_price(49000.00) # Notice the logger no longer reacts


[Ticker] BTC price updated to $48000.0
-> [Chart] Rendering new candle at $48000.0
   -> [Logger] INSERT INTO prices VALUES (48000.0)

[Ticker] BTC price updated to $51200.5
-> [Chart] Rendering new candle at $51200.5
   -> [Logger] INSERT INTO prices VALUES (51200.5)
   -> [SMS Alert] WHALE ACTIVITY! Price breached $50k ($51200.5)

[System] Disconnecting Logger for maintenance...

[Ticker] BTC price updated to $49000.0
-> [Chart] Rendering new candle at $49000.0
